# RAFT – Fine-tuning de Llama-3.1-8B-Instruct (Kaggle)

Entrena el modelo generador con el corpus RAFT (`raft_train.jsonl` / `raft_eval.jsonl`) y lo exporta en dos formatos:

| Salida | Para qué | Dónde queda |
|---|---|---|
| Adaptador LoRA (~170 MB) | Respaldo, reproducibilidad | `/kaggle/working/lora_raft` + HF `<usuario>/raft-acreditacion-8b-lora` |
| Modelo fusionado 16-bit (~16 GB) | API con vLLM (Modal) | HF `<usuario>/raft-acreditacion-8b` (privado) |
| GGUF Q4_K_M (~5 GB) | Ollama en la laptop | `/kaggle/working/gguf` + HF `<usuario>/raft-acreditacion-8b-gguf` (privado) |

Referencia del método: Zhang et al. (2024), *RAFT: Adapting Language Model to Domain Specific RAG*. RAFT es SFT estándar sobre datos (Q + D\* + distractores → respuesta CoT); aquí se hace con QLoRA por eficiencia.

## Antes de ejecutar (una sola vez)

1. **Subir los datos**: en Kaggle → *Datasets* → *New Dataset*, sube `raft_train.jsonl` y `raft_eval.jsonl` (carpeta `raft/salida`). Luego en este notebook: *Add Input* → tu dataset.
2. **Acelerador**: *Settings → Accelerator → GPU T4 x2* (la P100 **no** es compatible con Unsloth). Requiere cuenta verificada con teléfono.
3. **Internet**: *Settings → Internet → On*.
4. **Secretos** (*Add-ons → Secrets*): `HF_TOKEN` = token de Hugging Face con permiso **write** (https://huggingface.co/settings/tokens). Actívalo para este notebook.
5. Cambia `HF_USUARIO` en la celda de configuración.
6. Para que siga corriendo con el navegador cerrado: **Save Version → Save & Run All (Commit)**. Duración esperada: ~3–5 h en una T4 (dentro del límite de 12 h).

In [ ]:
# Usar una sola GPU (Unsloth entrena en una; evita problemas con T4 x2)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 1. Instalación

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --upgrade huggingface_hub

## 2. Configuración

In [ ]:
import glob, json, re, time, unicodedata
from pathlib import Path

HF_USUARIO = "TU_USUARIO_HF"          # <-- cambia esto
NOMBRE     = "raft-acreditacion-8b"

MODELO_BASE    = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"   # copia 4-bit no gated (licencia Llama 3.1)
MAX_SEQ_LENGTH = 3072      # el ejemplo más largo del corpus tiene ~2.600 tokens
SEMILLA        = 3407

# LoRA (valores estándar de QLoRA; el paper no usa LoRA, se declara como decisión de eficiencia)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.0

# Entrenamiento
EPOCAS        = 2
LR            = 2e-4
BATCH         = 2
GRAD_ACCUM    = 8          # batch efectivo = 16
WARMUP_RATIO  = 0.03
EVAL_CADA     = 100        # pasos

DIR_TRABAJO = Path("/kaggle/working")
DIR_TMP     = Path("/tmp/raft")          # disco temporal grande (el 16-bit no cabe en /kaggle/working, límite 20 GB)
DIR_TMP.mkdir(parents=True, exist_ok=True)

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_USUARIO != "TU_USUARIO_HF", "Configura HF_USUARIO"

REPO_LORA   = f"{HF_USUARIO}/{NOMBRE}-lora"
REPO_MERGED = f"{HF_USUARIO}/{NOMBRE}"
REPO_GGUF   = f"{HF_USUARIO}/{NOMBRE}-gguf"

## 3. Datos

In [ ]:
def buscar(nombre):
    rutas = glob.glob(f"/kaggle/input/**/{nombre}", recursive=True)
    assert rutas, f"No encontré {nombre}. ¿Agregaste el dataset como Input?"
    return rutas[0]

def leer_jsonl(ruta):
    with open(ruta, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_raw = leer_jsonl(buscar("raft_train.jsonl"))
eval_raw  = leer_jsonl(buscar("raft_eval.jsonl"))
print(f"train: {len(train_raw)}  eval: {len(eval_raw)}")
print([m["role"] for m in train_raw[0]["messages"]])
print(train_raw[0]["messages"][2]["content"][:500])

## 4. Modelo base + LoRA

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODELO_BASE,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,          # automático (fp16 en T4)
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = SEMILLA,
)
model.print_trainable_parameters()

## 5. Formato chat

Se aplica la plantilla de chat de Llama 3.1. Es la misma que quedará guardada con el modelo y que usarán vLLM y Ollama en inferencia, así que el formato de entrenamiento y el de uso coinciden.

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

def a_texto(ej):
    return {"text": tokenizer.apply_chat_template(ej["messages"], tokenize=False, add_generation_prompt=False)}

ds_train = Dataset.from_list(train_raw).map(a_texto, remove_columns=["messages"])
ds_eval  = Dataset.from_list(eval_raw).map(a_texto,  remove_columns=["messages"])

# Longitudes reales con el tokenizador de Llama; se descartan los que no caben (no debería haber)
def con_longitud(ds):
    return ds.map(lambda e: {"n_tokens": len(tokenizer(e["text"], add_special_tokens=False)["input_ids"])})
ds_train, ds_eval = con_longitud(ds_train), con_longitud(ds_eval)
import numpy as np
lt = np.array(ds_train["n_tokens"])
print(f"tokens/ejemplo  mediana={np.median(lt):.0f}  p95={np.percentile(lt,95):.0f}  max={lt.max()}")
antes = len(ds_train)
ds_train = ds_train.filter(lambda e: e["n_tokens"] <= MAX_SEQ_LENGTH)
ds_eval  = ds_eval.filter(lambda e: e["n_tokens"] <= MAX_SEQ_LENGTH)
print(f"descartados por longitud: {antes - len(ds_train)}")
print(ds_train[0]["text"][:1200])

## 6. Entrenamiento

La pérdida se calcula **solo sobre la respuesta del asistente** (`train_on_responses_only`). El modelo aprende a razonar y citar a partir de los documentos, no a reproducir los documentos.

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

cfg = dict(
    output_dir                  = str(DIR_TRABAJO / "checkpoints"),
    dataset_text_field          = "text",
    per_device_train_batch_size = BATCH,
    per_device_eval_batch_size  = BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_train_epochs            = EPOCAS,
    learning_rate               = LR,
    lr_scheduler_type           = "linear",
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = 0.01,
    optim                       = "adamw_8bit",
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = 10,
    eval_strategy               = "steps",
    eval_steps                  = EVAL_CADA,
    save_strategy               = "steps",
    save_steps                  = EVAL_CADA,
    save_total_limit            = 2,
    seed                        = SEMILLA,
    packing                     = False,
    report_to                   = "none",
)
# Compatibilidad entre versiones de TRL (max_seq_length -> max_length)
params = inspect.signature(SFTConfig.__init__).parameters
cfg["max_length" if "max_length" in params else "max_seq_length"] = MAX_SEQ_LENGTH
if "eval_strategy" not in params:
    cfg["evaluation_strategy"] = cfg.pop("eval_strategy")
args = SFTConfig(**cfg)

kw = dict(model=model, train_dataset=ds_train, eval_dataset=ds_eval, args=args)
if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters:
    kw["processing_class"] = tokenizer
else:
    kw["tokenizer"] = tokenizer
trainer = SFTTrainer(**kw)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

# Verificación del enmascarado: lo que se entrena debe empezar en "##Reason:"
ej = trainer.train_dataset[0]
etiquetas = [t for t in ej["labels"] if t != -100]
print("ENTRENA SOBRE →", tokenizer.decode(etiquetas)[:300])

In [ ]:
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}  {gpu.total_memory/1e9:.1f} GB")
t0 = time.time()
stats = trainer.train()
print(f"Entrenamiento: {(time.time()-t0)/60:.1f} min")

# Curvas de pérdida para la tesis
(DIR_TRABAJO / "log_entrenamiento.json").write_text(json.dumps(trainer.state.log_history, indent=2))
(DIR_TRABAJO / "config_entrenamiento.json").write_text(json.dumps({
    "modelo_base": MODELO_BASE, "max_seq_length": MAX_SEQ_LENGTH,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT},
    "epocas": EPOCAS, "lr": LR, "batch_efectivo": BATCH * GRAD_ACCUM,
    "n_train": len(ds_train), "n_eval": len(ds_eval),
    "minutos": round((time.time()-t0)/60, 1), "gpu": gpu.name,
}, indent=2))

In [ ]:
import matplotlib.pyplot as plt
h = trainer.state.log_history
tr = [(x["step"], x["loss"]) for x in h if "loss" in x]
ev = [(x["step"], x["eval_loss"]) for x in h if "eval_loss" in x]
plt.figure(figsize=(7,4))
plt.plot(*zip(*tr), label="train"); plt.plot(*zip(*ev), "o-", label="eval")
plt.xlabel("paso"); plt.ylabel("pérdida"); plt.legend(); plt.title("RAFT – pérdida")
plt.savefig(DIR_TRABAJO / "perdida.png", dpi=150, bbox_inches="tight"); plt.show()

## 7. Prueba rápida sobre eval

Esta celda es una verificación de sanidad, no la evaluación de la tesis (esa se hace con tu dataset de validación desde la app). Revisa en 20 ejemplos de eval:
- si el modelo produce `##Answer:`;
- si sus citas existen **literalmente** en alguno de los documentos que recibió.

In [ ]:
FastLanguageModel.for_inference(model)

def norm(s):
    s = unicodedata.normalize("NFKC", s or "").lower()
    return re.sub(r"[\W_]+", "", re.sub(r"<br\s*/?>", " ", s))

def generar(mensajes, max_new_tokens=512):
    ids = tokenizer.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

n, con_answer, citas_ok, citas_tot = 20, 0, 0, 0
for i, ej in enumerate(eval_raw[:n]):
    msgs = ej["messages"][:2]                      # system + user
    salida = generar(msgs)
    docs = norm(msgs[1]["content"])
    citas = re.findall(r"##begin_quote##(.*?)##end_quote##", salida, flags=re.S)
    con_answer += "##Answer:" in salida
    citas_tot += len(citas); citas_ok += sum(norm(c) in docs for c in citas if norm(c))
    if i < 3:
        print("PREGUNTA:", msgs[1]["content"].split("Pregunta:")[-1].strip())
        print("MODELO  :", salida[:700]); print("ESPERADO:", ej["messages"][2]["content"].split("##Answer:")[-1].strip()); print("-"*80)
print(f"Con ##Answer: {con_answer}/{n}   Citas literales: {citas_ok}/{citas_tot}")

## 8. Guardar el adaptador LoRA (pequeño; es el respaldo principal)

In [ ]:
from huggingface_hub import HfApi, login
login(token=HF_TOKEN)
api = HfApi()

model.save_pretrained(str(DIR_TRABAJO / "lora_raft"))
tokenizer.save_pretrained(str(DIR_TRABAJO / "lora_raft"))
model.push_to_hub(REPO_LORA, token=HF_TOKEN, private=True)
tokenizer.push_to_hub(REPO_LORA, token=HF_TOKEN, private=True)
print("LoRA →", REPO_LORA)

## 9. Modelo fusionado 16-bit → Hugging Face (para vLLM / Modal)

In [ ]:
api.create_repo(REPO_MERGED, private=True, exist_ok=True, token=HF_TOKEN)
dir_merged = DIR_TMP / "merged_16bit"
model.save_pretrained_merged(str(dir_merged), tokenizer, save_method="merged_16bit")
api.upload_folder(folder_path=str(dir_merged), repo_id=REPO_MERGED, token=HF_TOKEN,
                  commit_message="RAFT merged 16-bit")
print("Merged →", REPO_MERGED)
import shutil; shutil.rmtree(dir_merged, ignore_errors=True)   # liberar disco

## 10. GGUF Q4_K_M → Ollama

Unsloth compila llama.cpp para convertir el modelo; tarda entre 5 y 15 minutos. Si esta celda falla, no pierdes nada: el modelo fusionado ya quedó en HF y se puede convertir después.

In [ ]:
import shutil
dir_gguf = DIR_TMP / "gguf"
model.save_pretrained_gguf(str(dir_gguf), tokenizer, quantization_method="q4_k_m")

ggufs = sorted(glob.glob(f"{DIR_TMP}/**/*.gguf", recursive=True), key=os.path.getsize)
print(ggufs)
gguf = [g for g in ggufs if "q4_k_m" in g.lower()][-1]

destino = DIR_TRABAJO / "gguf"; destino.mkdir(exist_ok=True)
nombre_gguf = f"{NOMBRE}-Q4_K_M.gguf"
shutil.copy(gguf, destino / nombre_gguf)

# Modelfile para Ollama. num_ctx 4096 es OBLIGATORIO: los prompts RAFT llegan a ~2.600 tokens.
# Si Unsloth generó un Modelfile (con la plantilla TEMPLATE de Llama 3.1), se reutiliza.
mf_unsloth = glob.glob(f"{DIR_TMP}/**/Modelfile", recursive=True)
if mf_unsloth:
    cuerpo = Path(mf_unsloth[0]).read_text()
    cuerpo = re.sub(r"^FROM .*$", f"FROM ./{nombre_gguf}", cuerpo, count=1, flags=re.M)
    cuerpo = re.sub(r"^PARAMETER (temperature|num_ctx) .*\n?", "", cuerpo, flags=re.M)
else:
    cuerpo = f'FROM ./{nombre_gguf}\nPARAMETER stop "<|eot_id|>"\n'
(destino / "Modelfile").write_text(cuerpo.rstrip() + "\nPARAMETER temperature 0\nPARAMETER num_ctx 4096\n")
print((destino / "Modelfile").read_text())

api.create_repo(REPO_GGUF, private=True, exist_ok=True, token=HF_TOKEN)
api.upload_folder(folder_path=str(destino), repo_id=REPO_GGUF, token=HF_TOKEN, commit_message="RAFT GGUF Q4_K_M")
print("GGUF →", REPO_GGUF)

## 11. Resultado

En **Hugging Face** (repositorios privados):
- `…/raft-acreditacion-8b`: modelo fusionado, para desplegar en Modal con vLLM.
- `…/raft-acreditacion-8b-gguf`: GGUF y Modelfile, para Ollama.
- `…/raft-acreditacion-8b-lora`: adaptador LoRA, como respaldo.

En la pestaña **Output** de este notebook: `log_entrenamiento.json`, `config_entrenamiento.json` y `perdida.png` para la tesis, además de `gguf/` y `lora_raft/`.

**En tu laptop (Ollama)**, descarga `raft-acreditacion-8b-Q4_K_M.gguf` y el `Modelfile` en una misma carpeta y ejecuta:

```
ollama create raft-acreditacion -f Modelfile
ollama run raft-acreditacion
```

En la app usa `base_url="http://localhost:11434/v1"`, `api_key="ollama"` y `model="raft-acreditacion"`, junto con `raft_formato.construir_mensajes(...)`.